# 01B - Sequence Memory PyTorch Lab

**Guiding question:** can a reader finish at the last real note, ignore blank answers, and make guesses without peeking ahead?

> **What you finished last time:** [01A's notebook](01A-sequence-memory-theory.ipynb) built a running notecard and separated carried memory from its exposed hidden state.
> **What this notebook delivers:** one forecasting LSTM and one note-prediction LSTM, with visible checks for each batch contract.
> **What this chapter is not:** a music-quality benchmark or a tokenizer lesson.

Start with a smooth numeric signal so the question is easy to see: **read
24 observations, guess the next one**. Then return to 01A's piano notes.
The reader stays recurrent; the data changes from measured values to note IDs.

| Complaint | Smallest fix we'll check |
|---|---|
| Near-identical overlapping windows can make testing too easy | Keep validation answers later in time |
| A row of numbers doesn't tell the LSTM which axis means time | Batch, time, and feature seats |
| Notes need a numeric input, not pitch-sized ID arithmetic | Fixed addresses selecting trainable rows |
| Blank slots keep updating a short phrase's card | Stop that row at its real length |
| We stopped reading blanks, but can still grade blank answers | Skip pad targets in loss |
| An earlier guess could accidentally use a future note | Edit the future and compare earlier scores |
| A training gradient can be too large | Clip it after backward, before the optimizer step |

The [shared helper](sequence_helpers.py) keeps ordinary optimization steps in
one place. Each notebook runs from a fresh kernel; no earlier variables are needed.


### Embedding thread - use it now, open it up next

Keep the same map through the following notebooks:

```text
text -> pieces -> IDs -> fetched rows -> context processing -> scores -> checked answer
```

For piano notes, splitting on spaces already gives the pieces. Here the new
work is **reading fetched rows in order**, using an LSTM, and checking next-note
answers. Lookup supplies one compact row per note; training can make its
contents useful. The LSTM's changing memory belongs to the phrase, not to
the embedding table. Scores are a third object: possible answers.

**You will already train an embedding in this lab.** Returning to hand-picked
or random rows in the next theory notebook is a deliberate close-up, not a
claim that learning has not happened here. Each notebook starts fresh:
we carry the explanation forward, not saved weights or shared ID numbers.


In [ ]:
# -- Setup and canonical helpers --------------------------------------------------
import math
import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from sequence_helpers import backward_step, seed_everything, trainable_parameter_count

seed_everything(42)
torch.set_num_threads(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.facecolor": "#fffaf1", "axes.facecolor": "#fffaf1"})
print(f"Device: {device}")


## 1 - Read the recent past; guess one step ahead

A smooth wave gives us a deliberately easy first task. The lazy guess is
**repeat the last observed value**. We call that the persistence baseline.
Can a reader use the trend in the previous 24 values to do better?

![Two overlapping 24-value windows each point to their own next value. Training answer times end at 545, and validation answer times begin at 546.](images/01b-01-window-and-later-answer.png)

*The answer is outside its own window. Sliding one place changes both the
observed history and the answer we ask for.*

Split the **questions by answer time**, not by randomly dealing neighboring
windows to both piles. Our last training answer is at time 545; the first
validation answer is at 546. That is an actual later prediction.

The first validation history still overlaps earlier observations. That's
allowed: a real forecast can use the past it has already seen. It must not
read its own future answer. This is a chronological holdout, not independent
examples or a guarantee about generalizing to a different kind of signal.

**Predict:** with 24 observations in one question, is its answer position 23
or 24 when we count from zero? Which one must stay outside its input?


In [ ]:
# -- Low-noise series, sliding windows, and temporal split ------------------------
points = 720
time = torch.linspace(0, 8 * math.pi, points)
series = torch.sin(time) + 0.25 * torch.sin(3 * time)
window = 24

# Stack adds an example axis: 696 slices [24] -> [696,24]; scalar next values -> targets [696].
windows = torch.stack([series[i : i + window] for i in range(points - window)])
targets = torch.stack([series[i + window] for i in range(points - window)])
split = int(0.75 * len(windows))
X_train, X_valid = windows[:split], windows[split:]
y_train, y_valid = targets[:split], targets[split:]

print(f"Train windows: {tuple(X_train.shape)}")
print(f"Valid windows: {tuple(X_valid.shape)}")
print(f"Boundary indices: train ends at {split - 1}, validation starts at {split}")
last_training_answer_time = split - 1 + window
first_validation_answer_time = split + window
assert last_training_answer_time < first_validation_answer_time
assert torch.equal(X_valid[0], series[split:split + window])
assert torch.equal(y_valid[0], series[first_validation_answer_time])
print(f"Answer-time boundary: {last_training_answer_time} -> {first_validation_answer_time}")
print("Takeaway: validation answers are later; past observations may still overlap.")


In [ ]:
# -- Dataset and DataLoader: make one feature explicit ----------------------------
class WindowDataset(Dataset):
    def __init__(self, inputs, labels):
        # The LSTM needs features: [N,T] -> [N,T,1]; labels [N] -> [N,1] to match its scalar head.
        self.inputs = inputs.unsqueeze(-1).float()
        self.labels = labels.unsqueeze(-1).float()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=64, shuffle=True)
valid_loader = DataLoader(WindowDataset(X_valid, y_valid), batch_size=128, shuffle=False)
batch_x, batch_y = next(iter(train_loader))
print(f"Batch input: {tuple(batch_x.shape)} = (batch, time, feature)")
print(f"Batch target: {tuple(batch_y.shape)} = (batch, 1)")


### What does the LSTM actually receive?

`WindowDataset` puts a one-number feature row at each observed position.
The batch is a stack of questions: `[B, 24, 1]` means **B questions, 24
observations per question, one measured value per observation**. The loader
may shuffle training questions; it never scrambles positions inside a question.

The class below lets `nn.LSTM` read each question in order. It returns an
exposed hidden row after every position. `outputs[:, -1]` takes the final row
**within each question**, not the final question of the batch. A small linear
head turns each final row into one next-value guess.

All 24 seats are real here, so the last seat is safe. Keep that assumption
in mind; unequal note phrases will break it shortly.


In [ ]:
# -- First nn.LSTM model: predict from the final sequence output ------------------
class ForecastLSTM(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, values):
        # Read [B,T,1] -> [B,T,H]; take the last time step [B,H] and project H -> 1 per example.
        outputs, _ = self.lstm(values)
        return self.head(outputs[:, -1])

forecast_model = ForecastLSTM().to(device)
optimizer = torch.optim.Adam(forecast_model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()
train_history = []

for epoch in range(45):
    forecast_model.train()
    epoch_losses = []
    for inputs, labels in train_loader:
        predictions = forecast_model(inputs.to(device))
        loss = loss_fn(predictions, labels.to(device))
        backward_step(loss, forecast_model, optimizer)
        epoch_losses.append(float(loss.detach()))
    train_history.append(np.mean(epoch_losses))

forecast_model.eval()
valid_predictions = []
valid_targets = []
with torch.inference_mode():
    for inputs, labels in valid_loader:
        valid_predictions.append(forecast_model(inputs.to(device)).cpu())
        valid_targets.append(labels)
# Join batches along examples: lists of [B,1] -> [N,1] -> [N]; remove only the singleton head axis.
valid_predictions = torch.cat(valid_predictions).squeeze(1)
valid_targets = torch.cat(valid_targets).squeeze(1)
# Matching [N] arrays avoid accidental [N,N] broadcasting; mean reduces their squared errors to one score.
valid_mse = torch.mean((valid_predictions - valid_targets) ** 2).item()
baseline_mse = torch.mean((X_valid[:, -1] - y_valid) ** 2).item()

print(f"Parameters: {trainable_parameter_count(forecast_model):,}")
print(f"Persistence baseline MSE: {baseline_mse:.6f}")
print(f"LSTM validation MSE:      {valid_mse:.6f}")
if valid_mse < baseline_mse:
    print("This run beat repeating the last value on the later segment.")
else:
    print("This run did not beat persistence. Don't infer useful trend learning from training loss alone.")
assert np.isfinite(valid_mse)


In [ ]:
# -- Forecast evidence: loss and later-in-time predictions ------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_history, color="#60a5fa")
axes[0].set(title="Forecast training loss", xlabel="Epoch", ylabel="MSE")
axes[1].plot(valid_targets[:120], label="actual", color="#34d399")
axes[1].plot(valid_predictions[:120], label="predicted", color="#f97316")
axes[1].set(title="Later validation segment", xlabel="Step")
axes[1].legend()
plt.show()


### Equal windows worked. Shorter phrases expose a new problem.

We read measured values and made one guess at the end of each window. Now
read short piano-note phrases and make a **next-note guess at every real
position**. This deliberately small task tests the pipeline, not musical skill.

The familiar `D3 A3 D4` grows into **D3 A3 D4 F4 A4**. Another phrase has
only three notes. A rectangular batch needs blank seats beside that short
phrase, but its reader must not mistake those seats for more music.


### Preview checkpoint - use these tensors now; explain them next

This section needs a language-shaped sequence so you can test recurrent memory. It therefore uses
four contracts before Chapter 02 formally derives them:

| Contract used in this lab | What you need here | What Chapter 02 will add |
|---|---|---|
| Token IDs | Integers retrieve embedding rows | How BPE pieces and vocabulary addresses are created |
| Shifted targets | Position `t` predicts the next token | Why logits and target positions must align |
| Padding | Unequal rows share one batch shape | Which positions are bookkeeping and how loss ignores them |
| Causal invariance | A future edit cannot change an earlier output | How causal access differs from padding and loss masks |

**Predict before continuing:** If two sequences share the same prefix but have different final
tokens, which earlier logits may change in a unidirectional LSTM? Keep that answer; the causal
invariance section will test it.

You are not expected to know how the vocabulary was built yet. The next chapter returns to this
table and makes each representation contract explicit.

## 2 - Keep the answer one note ahead

Take **D3 A3 D4 F4 A4**. At `D4`, the reader has seen `D3`, `A3`, and
`D4`; ask it to guess `F4`. The full note list supplies both sides of the
practice question. We don't need someone to write extra labels.

![D3 A3 D4 F4 are the input positions, with A3 D4 F4 A4 directly below as their one-step-ahead answers.](images/01b-02-shift-the-answer-not-the-question.png)

*Compare each input position with the answer directly beneath it. The two
rows are aligned for checking, not both handed to the reader at that position.*

`row[:-1]` leaves the last note out of the input. `row[1:]` starts the
answer row one note later. The lengths match; the contents are offset.
We do this **before padding**, so no blank becomes a real next-note answer.

**Predict:** a five-note phrase makes how many next-note questions: four
or five? At the first `D3`, is the checked answer `D3` or `A3`?


A **token** is just the piece we handle as one unit; here one named piano
note is one token. Its **ID** is a registry number, not a pitch measurement.
Changing `D3`'s ID doesn't transpose the tune.

Short rows get `<pad>` placeholders to fit the longest input row. `lengths`
records how many real inputs each row has. `mask` marks those same seats with
True. These describe the input rectangle; neither automatically tells the
loss to ignore blank **answers**.

[02A's notebook](../02-tokenization-and-embeddings/02A-tokenization-and-embeddings-theory.ipynb)
will derive pieces and addresses. For now, the question is what happens to
the reader's memory once we put those addresses into a batch.


In [ ]:
# -- Token IDs, shifted targets, and padding -------------------------------------
PAD = "<pad>"
motifs = [
    "D3 A3 D4 F4 A4",
    "D3 A3 D4 F4",
    "D3 A3 C4",
    "D3 F3 A3 D4 F4 A4",
]
vocabulary = [PAD] + sorted({token for motif in motifs for token in motif.split()})
token_to_id = {token: index for index, token in enumerate(vocabulary)}
pad_id = token_to_id[PAD]

encoded = [torch.tensor([token_to_id[token] for token in motif.split()]) for motif in motifs]
# Shift each [L] motif into two [L-1] rows: every input is paired with its next-note answer.
inputs = [row[:-1] for row in encoded]
targets = [row[1:] for row in encoded]
lengths = torch.tensor([len(row) for row in inputs])
# Unequal [L_i] rows -> [B,T_max] rectangles, in the same row order for inputs and targets.
padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=pad_id)
padded_targets = pad_sequence(targets, batch_first=True, padding_value=pad_id)
mask = padded_inputs.ne(pad_id)

print(f"Vocabulary: {vocabulary}")
print(f"Inputs shape:  {tuple(padded_inputs.shape)}")
print(f"Targets shape: {tuple(padded_targets.shape)}")
print(f"Lengths: {lengths.tolist()}")
print("First input/target pair:")
print(" ", [vocabulary[index] for index in padded_inputs[0]], "->",
      [vocabulary[index] for index in padded_targets[0]])
assert lengths.tolist() == [len(row) - 1 for row in encoded]
assert torch.equal(mask, padded_targets.ne(pad_id))
assert int(mask.sum()) == sum(lengths.tolist())
print(f"Real next-note questions: {mask.sum().item()}; rectangular seats: {mask.numel()}")


### An address fetches numbers. The numbers feed the reader.

`nn.Embedding` stores one short row per note. We pass fixed integer IDs to
it; it returns the stored rows. Those rows start random once, then training
can change their values because they are model weights. The IDs stay fixed.
Right now this table hasn't learned anything yet.

For this preview each row has six numbers. A batch of IDs `[B, T]` becomes
`[B, T, 6]`: one six-number feature row for every seat. Fetching a row
doesn't summarize a phrase; the LSTM will build the changing notecard.

`padding_idx` keeps the blank input row from receiving lookup gradients.
It does **not** switch off the LSTM's recurrent update. A zero input can
still change a nonzero old state. That is the next failure to inspect.


**Experiment state - note lookup, not phrase memory**

| Experiment | Vocabulary and stored rows | Training state |
|---|---|---|
| Lookup cell immediately below | Local note IDs; six numbers per row | Random at creation; no optimizer step in this preview |
| Complete `MotifLSTM` later | Same note vocabulary; its own twelve-number rows | Fresh initialization; embedding, LSTM, and head train together |

The six-number preview is not the complete model's table. In both cases,
an ID selects a stored row; it is never used as a pitch-sized measurement.


In [ ]:
# -- Embedding lookup: IDs are addresses, not numeric magnitudes -----------------
embedding = nn.Embedding(len(vocabulary), embedding_dim=6, padding_idx=pad_id)
# Lookup adds features, not meaning to the ID magnitude: [B,T] -> [B,T,6].
embedded = embedding(padded_inputs)
selected_row = embedding.weight[padded_inputs[0, 1]]
exact_lookup = torch.equal(embedded[0, 1], selected_row)

print(f"(B, T) IDs -> (B, T, D) vectors: {tuple(embedded.shape)}")
print(f"Lookup equals selected table row: {exact_lookup}")
print("Takeaway: token ID 5 is an address; it is not five times token ID 1.")
assert exact_lookup


## 3 - A blank seat must not become another reading step

Our shortest phrase is **D3 A3 C4**, so its input is just **D3 A3**.
We pad that input to five seats. If the LSTM reads all five, its returned
final card comes **three updates after A3**, not immediately after A3.
Zeros aren't an instruction to stop.

![The D3 A3 input is followed by three blank seats. Unpacked recurrence keeps updating; packing stops at A3. A separate loss mask skips blank targets.](images/01b-03-stop-reading-skip-grading.png)

*Read the two input tracks first. Packing changes where each reader stops.
The two bottom cards name different jobs, not interchangeable ways to solve one job.*

**Packing** hands PyTorch the real lengths. It schedules only real reading
steps, without joining unrelated phrases. Internally it may reorder work;
unpacking restores the original batch order.

Let's first read the padded rectangle **without** packing and compare its
short row's final card with reading only that row's real notes. Then pack
the same inputs through the **same weights** and check again.

**Predict:** should the unpacked final card match the two-note reading?
Should the packed one match? What should occupy blank output seats after
we restore the rectangle?


In [ ]:
# -- Packed sequences: final state means final real token -------------------------
inspection_lstm = nn.LSTM(input_size=6, hidden_size=8, batch_first=True)
short_row = lengths.argmin().item()
short_length = lengths[short_row].item()
with torch.inference_mode():
    _, (plain_final, _) = inspection_lstm(embedded)
    _, (solo_final, _) = inspection_lstm(embedded[short_row:short_row + 1, :short_length])
plain_gap = (plain_final[-1, short_row] - solo_final[-1, 0]).abs().max().item()
assert plain_gap > 1e-6
print(f"Without packing: extra blank steps changed the short row's final card by {plain_gap:.6f}")
# Pack [B,T,6] -> sum(lengths) real rows [N_real,6] plus the schedule that keeps sequences separate.
packed = pack_padded_sequence(
    embedded,
    lengths.cpu(),
    batch_first=True,
    enforce_sorted=False,
)
# Real rows now have width 8; h_n/c_n [1,B,8] retain each sequence's final real state.
packed_outputs, (h_n, c_n) = inspection_lstm(packed)
# Restore [B,T,8] and the original batch order so lengths and masks still line up.
unpacked, unpacked_lengths = pad_packed_sequence(
    packed_outputs,
    batch_first=True,
    total_length=padded_inputs.size(1),
)

# [B,T] -> [B,T,1] broadcasts the pad test across all 8 features; selected values become one flat vector.
padded_values = unpacked.masked_select(~mask.unsqueeze(-1))
print(f"Unpacked outputs: {tuple(unpacked.shape)}")
print(f"Final hidden:     {tuple(h_n.shape)} = (layers, batch, hidden)")
print(f"Final cell:       {tuple(c_n.shape)}")
print(f"Returned lengths: {unpacked_lengths.tolist()}")
print(f"Maximum absolute padded output: {padded_values.abs().max().item():.6f}")
assert torch.allclose(padded_values, torch.zeros_like(padded_values))


In [ ]:
# -- Verify final hidden equals each row's last real output -----------------------
row_indices = torch.arange(len(lengths))
# Pair each row with its own last real time index: [B,T,8] -> [B,8], matching h_n[-1].
last_real_outputs = unpacked[row_indices, lengths - 1]
final_state_match = torch.allclose(last_real_outputs, h_n[-1], atol=1e-6)
print(f"Final hidden matches each last real timestep: {final_state_match}")
print("Takeaway: packing protects sequence meaning, not just compute.")
assert final_state_match
packed_gap = (h_n[-1, short_row] - solo_final[-1, 0]).abs().max().item()
assert packed_gap < 1e-6
print(f"With packing: final card differs from the solo real-note reading by {packed_gap:.8f}")


## 4 - We stopped the reader. We still have to mark the right answers.

The LSTM produces an exposed hidden row at each real note. A shared prediction
head turns each row into **one raw score per vocabulary item**, called logits.
The same head is used at every position.

```text
note IDs [B, T]
-> fetched rows [B, T, D]
-> packed reader; restore output seats [B, T, H]
-> one score sheet per seat [B, T, V]
```

Read the last shape as **B phrases, T seats per phrase, V possible answers
on each seat's score sheet**. No guessing or averaging across phrases happens.
The vocabulary axis lists possible answers, not another set of time positions.

Packing gave blank output seats zeros. The prediction head can still turn
those zeros into scores through its bias. Even zero scores would not make
those seats real questions. We need a separate rule: **grade only real answers**.


In [ ]:
# -- Tiny token model: embedding -> packed LSTM -> vocabulary logits -------------
class MotifLSTM(nn.Module):
    def __init__(self, vocab_size, pad_id, embedding_dim=12, hidden_size=20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids, sequence_lengths):
        # IDs [B,T] -> rows [B,T,D]; packing drops blank timesteps, not real sequence boundaries.
        vectors = self.embedding(token_ids)
        packed = pack_padded_sequence(
            vectors,
            sequence_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, state = self.lstm(packed)
        # Unpack real hidden rows [N_real,H] -> [B,T,H] to align predictions with the padded answer grid.
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=token_ids.size(1),
        )
        # The same H -> V projection at every position gives [B,T,V] raw word scores.
        return self.head(outputs), state

token_model = MotifLSTM(len(vocabulary), pad_id).to(device)
logits, state = token_model(padded_inputs.to(device), lengths)
print(f"Logits: {tuple(logits.shape)} = (batch, time, vocabulary)")
print(f"Hidden state: {tuple(state[0].shape)}")


### Packing isn't a loss mask

Flatten the batch and position axes in the same order on both sides: each
score sheet stays beside its own answer. Then exclude target IDs equal to
`<pad>` and average over only the real answers.

`ignore_index=pad_id` does that exclusion for cross-entropy. The explicit
calculation below must give the same result. There are 14 real questions
in 20 rectangular seats; dividing by 20 would grade the six blank answers too.

The wrong average can be higher or lower. The problem isn't which number
looks nicer; it's measuring a different task.


In [ ]:
# -- Wrong versus correct loss: padding is bookkeeping, not a target -------------
# Cross-entropy wants one row per answer: [B,T,V] -> [B*T,V], targets [B,T] -> [B*T], same order.
flat_logits = logits.reshape(-1, len(vocabulary))
flat_targets = padded_targets.to(device).reshape(-1)
per_position = nn.functional.cross_entropy(
    flat_logits,
    flat_targets,
    reduction="none",
)
# Per-position loss [B*T] -> scalar; this wrong mean includes the padded slots.
wrong_loss = per_position.mean()

real_target_mask = flat_targets.ne(pad_id)
# Select [N_real] losses and divide by N_real, not B*T, so blanks cannot water down the score.
explicit_masked_loss = per_position[real_target_mask].sum() / real_target_mask.sum()
ignore_index_loss = nn.functional.cross_entropy(
    flat_logits,
    flat_targets,
    ignore_index=pad_id,
)

print(f"Wrong all-position mean:       {wrong_loss.item():.6f}")
print(f"Explicit real-token reduction: {explicit_masked_loss.item():.6f}")
print(f"ignore_index reduction:        {ignore_index_loss.item():.6f}")
print(f"Correct reductions match: {torch.allclose(explicit_masked_loss, ignore_index_loss)}")
assert torch.allclose(explicit_masked_loss, ignore_index_loss)


In [ ]:
# -- More blank targets must not change the score on real questions --------------
extra_targets = torch.cat(
    [padded_targets.to(device),
     torch.full((len(lengths), 2), pad_id, dtype=torch.long, device=device)], dim=1,
)
extra_logits = torch.cat([logits.detach(), logits.new_zeros(len(lengths), 2, len(vocabulary))], dim=1)
more_padding_loss = nn.functional.cross_entropy(
    extra_logits.reshape(-1, len(vocabulary)), extra_targets.reshape(-1), ignore_index=pad_id,
)
assert torch.allclose(more_padding_loss, ignore_index_loss)
print("Add two blank columns:", ignore_index_loss.item(), "->", more_padding_loss.item())
print("Takeaway: extra bookkeeping seats don't change the loss on real answers.")


## 5 - Could the reader be peeking at a later note?

A left-to-right LSTM should not. But rather than trusting the name, edit one
later input and check its earlier guesses.

![The original D3 A3 D4 F4 input and edited D3 A3 D4 C4 input share their first three notes. Score rows at those three positions must match, while the edited-position scores may differ.](images/01b-04-future-edit-same-prefix.png)

*The teal cards have identical histories. The orange cards don't. Compare
score rows, not just the winning note; two different distributions can have
the same winner.*

At the `D4` position, the model guesses the next note without receiving
either `F4` or `C4`. Changing that later input must not change this earlier
guess. At the edited position itself, different scores are allowed.

We use evaluation mode and the same weights for both runs. The check measures
**causal access**, not musical accuracy. No separate triangular attention mask
is needed for this unidirectional recurrence; it is not reading backward.

**Predict:** should the first three score rows change? What about the fourth?


In [ ]:
# -- Causal-invariance proof: future edits cannot change earlier logits ----------
causal_model = MotifLSTM(len(vocabulary), pad_id).to(device).eval()
reference = padded_inputs[:1].clone()
changed_future = reference.clone()
last_index = lengths[0].item() - 1
changed_future[0, last_index] = token_to_id["C4"]

with torch.inference_mode():
    reference_logits, _ = causal_model(reference.to(device), lengths[:1])
    changed_logits, _ = causal_model(changed_future.to(device), lengths[:1])

prefix_difference = (
    reference_logits[:, :last_index] - changed_logits[:, :last_index]
).abs().max().item()
edited_position_difference = (
    reference_logits[:, last_index] - changed_logits[:, last_index]
).abs().max().item()

print(f"Maximum earlier-logit difference: {prefix_difference:.8f}")
print(f"Edited-position difference:       {edited_position_difference:.8f}")
print("Takeaway: future token changes leave earlier unidirectional logits invariant.")
assert prefix_difference < 1e-7
assert edited_position_difference > 0


## 6 - Clip an oversized teaching signal before using it

We have kept the reading steps, answer checking, and future access honest.
One practical problem remains: a recurrent gradient can occasionally be huge.
Before applying it, put a ceiling on its overall size.

```text
guess -> check real answers -> backward -> measure and clip gradients -> optimizer step
```

Clipping rescales an oversized combined gradient; it doesn't change note
IDs, truncate the sequence, or erase the model's memory. A small gradient
stays small. The ceiling is a safety brake, not a cure for vanished feedback.

The helper clears old gradients, calls `backward()`, measures and clips,
then calls `step()`. Before backward there is no new gradient to cap; after
step the weights have already changed.

**Predict:** if every measured norm is below the ceiling, should clipping
change any update? Check the actual count; don't assume it activated.


In [ ]:
# -- A controlled oversized gradient makes the brake visible -------------------
clip_probe = nn.Parameter(torch.zeros(2))
probe_weights_before = clip_probe.detach().clone()
(clip_probe * torch.tensor([3.0, 4.0])).sum().backward()
assert clip_probe.grad is not None
probe_gradient_before = clip_probe.grad.clone()
probe_norm_before = torch.nn.utils.clip_grad_norm_([clip_probe], max_norm=1.0)
probe_norm_after = clip_probe.grad.norm()
assert torch.allclose(probe_norm_before, torch.tensor(5.0))
assert probe_norm_after <= 1.0
assert torch.allclose(clip_probe.grad, probe_gradient_before / 5, atol=1e-6)
assert torch.equal(clip_probe.detach(), probe_weights_before)
print("Gradient before / after:", probe_gradient_before.tolist(), clip_probe.grad.tolist())
print(f"Combined norm: {probe_norm_before.item():.1f} -> {probe_norm_after.item():.6f}")
print("Takeaway: clipping scaled the gradient, not the weights. No optimizer step has happened.")


In [ ]:
# -- Full token training loop with visible pre- and post-clip norms --------------
seed_everything(42)
token_model = MotifLSTM(len(vocabulary), pad_id).to(device)
optimizer = torch.optim.Adam(token_model.parameters(), lr=0.03)
loss_history = []
norm_history = []
clip_norm = 1.0

for epoch in range(120):
    token_model.train()
    logits, _ = token_model(padded_inputs.to(device), lengths)
    # Flatten only batch/time: [B,T,V] -> [B*T,V], answers [B,T] -> [B*T]; ignore_index skips blanks.
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, len(vocabulary)),
        padded_targets.to(device).reshape(-1),
        ignore_index=pad_id,
    )
    pre_clip, post_clip = backward_step(loss, token_model, optimizer, clip_norm)
    loss_history.append(float(loss.detach()))
    norm_history.append((pre_clip, post_clip))

clipped_steps = sum(pre > clip_norm for pre, _ in norm_history)
print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Final loss:   {loss_history[-1]:.6f}")
print(f"Steps above clip threshold: {clipped_steps}/{len(norm_history)}")
print(f"Final recorded norms: pre={norm_history[-1][0]:.4f}, post={norm_history[-1][1]:.4f}")
print("The brake activated on these steps only; zero means this run never needed that ceiling.")
assert loss_history[-1] < loss_history[0]


### Your turn - move the ceiling, without retraining

Change only `clip_norm_to_test`. Predict whether more or fewer **recorded**
steps would exceed that ceiling. This rechecks the saved pre-clip norms;
it doesn't simulate the different training path that retraining could create.


In [ ]:
# -- Your turn: inspect one known training control --------------------------------
# CHANGE THIS: try 0.25 or 2.0
clip_norm_to_test = 0.5
would_clip = sum(pre > clip_norm_to_test for pre, _ in norm_history)
print(f"At threshold {clip_norm_to_test:.2f}, {would_clip}/{len(norm_history)} recorded steps would clip.")
print("Check: lowering the threshold cannot reduce the number of clipped recorded steps.")
if clip_norm_to_test <= clip_norm:
    assert would_clip >= clipped_steps
else:
    assert would_clip <= clipped_steps


### The same training loop, a different job for each object

| Object here | Larger language-model counterpart | Same underlying job |
|---|---|---|
| Small note-ID table | Larger word/subword embedding table | Fetch a trainable row using a fixed ID |
| One-layer note LSTM | Wider or stacked recurrent reader | Update carried state in input order |
| Real lengths for four phrases | Unequal documents in a batch | Keep blank seats from extending the real sequence |
| One score per note ID | One score per vocabulary token | Align each score sheet with its next-token answer |
| `ignore_index` for pad targets | Token-level loss masking | Grade only real prediction tasks |

**Practical failure modes:** calling zero-valued input a stop signal; assuming
packing also masks the loss; selecting the last rectangular seat for a short
sequence; flattening predictions and answers in different orders; trusting
causality without checking a future edit.

Next, [01C's notebook](01C-cinematic-piano-memory.ipynb) keeps the reader and
training helper, but makes the memory question audible: two phrases share a
clock and differ only in their opening key cue. Does their ending change?

[02A's notebook](../02-tokenization-and-embeddings/02A-tokenization-and-embeddings-theory.ipynb)
then slows down the pieces, IDs, and row lookup previewed here.


## 7 - Scorecard and handoff

| Contract | Evidence |
|---|---|
| temporal validation | answer times are later; overlap in already observed history is explicit |
| `(B, T, F)` | forecast batches expose all three axes |
| shifted targets | each input token predicts the next token |
| packed sequences | padded final state drifts; packed final state matches the solo real-note reading |
| padding-aware loss | explicit reduction equals `ignore_index`; extra blank columns don't change it |
| causal recurrence | future edit leaves earlier logits unchanged |
| clipping order | pre- and post-clip norms are recorded |

**Built and measured:** forecasting, token IDs, embeddings, packing, vocabulary logits, masked loss, causal invariance, clipping.

**Explained or illustrated:** why temporal splits matter and why padding is bookkeeping.

**Named and out of scope:** bidirectional recurrence, attention, and production checkpointing.

Next, [01C](01C-cinematic-piano-memory.ipynb) reuses the canonical helper contract and asks you to hear the memory difference.


### Carry forward - what exactly did lookup do, and what could teach its rows?

For `D3 A3 D4 F4 A4`, fixed note IDs selected rows; the LSTM read those
rows in order; shifted answers supplied feedback to the whole model.
Lookup did not itself remember the phrase.

[02A](../02-tokenization-and-embeddings/02A-tokenization-and-embeddings-theory.ipynb)
switches from piano notes to pet-sitter text because spelling now makes
choosing pieces a real problem. The lookup contract does not change.
Two named examples take over: the **lookup microscope**, `the ___ sat`,
isolates row selection; the **learning game**, `cat/dog -> sat/slept`
and `bird -> flew/sang`, asks what could make those rows useful.
These are new local vocabularies, not a conversion of the trained note table.
